[<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/snntorch_alpha_w.png?raw=true' width="400">](https://github.com/jeshraghian/snntorch/)




#Connection to the drive and loading of libraries.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install snntorch --quiet

In [3]:
import numpy as np
print(np.__version__)

2.1.3


In [4]:
# imports
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import torch.nn.functional as F

from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import itertools
import random
import statistics
import tqdm

import scipy.io as sio
import os
import requests


In [5]:
def load_dataset(train_setting,path):
  var_dict = 'broadband_dataset'
  # neural_chunks
  mat_contents = sio.loadmat(path)
  neural_chunks = mat_contents[var_dict][0,0]['neural_chunks']
  outs = mat_contents[var_dict][0,0][train_setting['target']]
  return path, mat_contents, var_dict, neural_chunks, outs


def CC(gt, inference):
  return np.corrcoef(gt, inference)[0, 1]

def Rsquare(gt, inference):
    num = np.square(gt - inference)
    den = np.square(gt - np.mean(gt))
    return 1 - np.sum(num)/np.sum(den)


def eval_len_window(reaches, n_rch_train):
  target_training_n = reaches[::n_rch_train]
  return int(np.mean(np.diff(target_training_n)))


def format_broadband(spike, vel, reaches, n_rch_train, len_window,ch):
  target_training_n = reaches[::n_rch_train]
  target_training_n = target_training_n - target_training_n[0]
  len_broad = len(spike)
  n_window = 0
  for start in target_training_n:
    if start + len_window < len_broad:
      n_window = n_window + 1
  spike = np.transpose(spike, (1, 0))
  vel   = np.transpose(vel, (1, 0))
  spike_w = np.zeros((n_window, ch, len_window))
  vel_w   = np.zeros((n_window,  2, len_window))
  for i, start in enumerate(target_training_n):
    if start + len_window > len_broad:
      break
    spike_w[i,:,:] = spike[:, start:start + len_window]
    vel_w[i,:,:]   =   vel[:, start:start + len_window]
  for finestra in range(n_window):
    start_broadband = target_training_n[finestra]
    check1 = spike_w[finestra, :,:] - spike[:, start_broadband:start_broadband+len_window]
    check2 =   vel_w[finestra, :,:] -   vel[:, start_broadband:start_broadband+len_window]
    if np.max(check1)> 0 or np.max(check2)> 0:
      print("errore")
      break
  return spike_w, vel_w

#Training setting

In [24]:
train_setting = {
    'target' : 'vel',                         #pos; vel; pol; pol_var; pol_prof
    'decay_train'     :True,                  #True: voltage decay trained;
    'thr_train'       :False,                 #True: threshold trained;
    'bias'            :False,                 #True: inserimento di un bias in ogni neurone
    '1beta4layer'     :False,                 #True: un solo valore di decay per layer
    'beta_out_single' :False,                 #True: un solo valore di decay per i neuroni dell'ultimo layer
    '1thr4layer'      :True,                  #True: un solo valore di threshold per layer
    'lr' : 1e-3,                              #learning rate
    'epochs': 300,                            #epoche
    'batch': 10,                              #batch size
    'hyperparams': [64, 128, 64],             #rete implementata 3 layer
    'thr_init' : 0.1,                         #threshold costante
    'bet_init': 0.9,                          #beta decay costante
    'reset': "subtract",                      #none; subtract; zero; meccanismo di reset dei neuroni
}

patience=50
trained_folder =  '/content/'                 ## set destination folder where trained model will be saved

if not os.path.exists(trained_folder):
        os.makedirs(trained_folder)
        print(f'Mkdir {trained_folder}.')

#Loading dataset


In [7]:
training_ratio = [0.8, 0.1, 0.1]
dataset_path = '/content/indy_20170123_02_1ms_thr5_mean_abs_window8192.mat' # change to set different path or select different record
path, mat_contents, var_dict, neural_chunks, uscite = load_dataset(train_setting,dataset_path)
input_neurons = neural_chunks.shape[0]

In [ ]:
neural_chunks = np.transpose(neural_chunks, (1,0))
target = mat_contents[var_dict][0,0]['index_var_targets']
target = target.squeeze()

start_reach = 0
start       = 0

# N reaches per set
training_reaches   = int(len(target)*training_ratio[0])
validation_reaches = int(round(len(target)*training_ratio[1]))
test_reaches       = int(len(target) - validation_reaches - training_reaches)

# Index of i reach
training_entries   = int(target[training_reaches])
validation_entries = int(target[training_reaches + validation_reaches])

print(path)

print("n tot reaches : {}".format(len(target)))
print("\nreaches training : {}".format(training_reaches -start_reach))
print("reaches validation : {}".format(validation_reaches))
print("reaches test : {}".format(len(target) - training_reaches - validation_reaches))


print("\nTraining   window : {}:{}".format(start, training_entries))
print("Validation window : {}:{}".format(training_entries, validation_entries))
print("Test       window : {}:{}".format(validation_entries, len(uscite)))

training_input   = neural_chunks[start:training_entries, :]
validation_input = neural_chunks[training_entries: validation_entries, :]
test_input       = neural_chunks[validation_entries:, :]

training_output = uscite[start:training_entries, :]
validation_output = uscite[training_entries: validation_entries, :]
test_output = uscite[validation_entries:, :]

print(f"\n tot samples training : {training_input.shape[0]}")
print(f"n tot samples  validation : {validation_input.shape[0]}")
print(f"n tot samples  testing  : {test_input.shape[0]}\n")

n_reac = 3 # training segments include 3 reaches on average
len_window = eval_len_window(target[0:training_reaches], n_reac)
training_input, training_output     = format_broadband(training_input, training_output, target[start_reach:training_reaches], n_reac, len_window,training_input.shape[1])
validation_input, validation_output = format_broadband(validation_input, validation_output, target[training_reaches:training_reaches + validation_reaches], n_reac, len_window,training_input.shape[1])

print(f"training_input {training_input.shape}")
print(f"training_output {training_output.shape}\n")

print(f"validation_input {validation_input.shape}")
print(f"validation_output {validation_output.shape}\n")

print(f"test_input {test_input.shape}")
print(f"test_output {test_output.shape}")

plt.figure(figsize = (20, 5))
plt.plot(uscite[:, 0])
plt.axvline(x=start, color='r', linestyle='--')
plt.axvline(x=training_entries, color='r', linestyle='--')
plt.axvline(x=validation_entries, color='r', linestyle='--')

In [ ]:
es = training_output[1][0]
plt.plot(es)

In [ ]:
def running_sum(signal, window):
    cumsum = np.cumsum(np.insert(signal, 0, 0))
    result = cumsum[window:] - cumsum[:-window]
    return result

R_x = []
Cc_x = []
R_y = []
Cc_y = []

check = torch.tensor(training_input.transpose(1,0,2)).flatten(1,2)
print(check.shape)

vel_GT = torch.tensor(training_output.transpose(1,0,2)).flatten(1,2).numpy()
print(vel_GT.shape)

for c in range(check.shape[0]):
  s = running_sum(check[c], 100).numpy()
  R_x.append(Rsquare(vel_GT[0][99:],s))
  Cc_x.append(CC(vel_GT[0][99:],s))
  R_y.append(Rsquare(vel_GT[1][99:],s))
  Cc_y.append(CC(vel_GT[1][99:],s))

for c in range(check.shape[0]):
  print(f'Channel {c} CCx {Cc_x[c]} CCy {Cc_y[c]}\n')

Rx = np.array(R_x)
Cc_x = np.array(Cc_x)
Ry = np.array(R_x)
Cc_y = np.array(Cc_y)

R = abs(Rx) + abs(Ry)
Cc = abs(Cc_x) + abs(Cc_y)

print(R.shape, Cc.shape)

meanCC = np.median(Cc)
print('Threshold for channel selection',meanCC)

selectedx = list(np.where(abs(Cc_x)>meanCC)[0])
selectedy = list(np.where(abs(Cc_y)>meanCC)[0])
newy = [selectedy[i] for i in range(len(selectedy)) if selectedy[i] not in selectedx]
selected = np.array(selectedx + newy)

print('Selected channels:')
print(selected)
print(len(selected))

## channel selection
print('Data shape before channel selection')
print(training_input.shape,validation_input.shape,test_input.shape)

training_input = training_input.transpose(1,0,2)[selected].transpose(1,0,2)
validation_input = validation_input.transpose(1,0,2)[selected].transpose(1,0,2)
test_input = test_input.transpose(1,0)[selected].transpose(1,0)

print('Data shape after channel selection')
print(training_input.shape,validation_input.shape,test_input.shape)

#Saving train setting

In [11]:
import datetime
data = str(datetime.datetime.now())
data = ((data[0:16].replace(" ", "_")).replace("-","_")).replace(":","_")

select_seed = 17
# Seed
torch.manual_seed(select_seed)
random.seed(select_seed)
np.random.seed(select_seed)

with open(trained_folder + "/0parametri_training.txt", "w") as f_save:
  f_save.write(path + '\n')
  for chiave in train_setting:
    f_save.write("{} = {} \n".format(chiave, train_setting[chiave]))
  f_save.write('Random seed{} \n'.format(select_seed))


#Create Datasets

In [ ]:
class BroadbandDataset(torch.utils.data.Dataset):
    def __init__(self, target, spike_in):
      self.labels   = target
      self.features = spike_in
    def __len__(self):
        """Number of samples."""
        return len(self.labels)
    def __getitem__(self, idx):
        """General implementation, but we only have one sample."""
        batch_feature = self.features[idx, :]
        batch_labels  = self.labels[idx, :]
        return (
            torch.tensor(batch_feature.astype(np.float32)).unsqueeze(0),
            torch.tensor(batch_labels.astype(np.float32))
        )

class RegressionDataset(torch.utils.data.Dataset):
    def __init__(self, target, spike_in):
        self.num_samples = target.shape[0]
        timesteps = target.shape[2]
        target   = target.astype(np.float32)
        spike_in = spike_in.astype(np.float32)
        labels_target = []
        lista = []
        for i in range(self.num_samples):
          target_i = target[i, :, :]
          target_i = np.transpose(target_i, (1, 0))
          labels_target.append(torch.from_numpy(target_i))
          spike_ing = spike_in[i, :, :]
          spike_ing = np.transpose(spike_ing, (1, 0))
          lista.append(torch.from_numpy(spike_ing))
        self.labels   = torch.stack(labels_target, dim=1)
        self.features = torch.stack(lista, dim=1)
    def __len__(self):
        """Number of samples."""
        return self.num_samples
    def __getitem__(self, idx):
        """General implementation, but we only have one sample."""
        return self.features[:, idx, :], self.labels[:, idx, :], idx

batch_size = train_setting['batch']
training_dataset = RegressionDataset(target=training_output, spike_in = training_input)
train_loader = torch.utils.data.DataLoader(dataset=training_dataset, batch_size=batch_size, drop_last=False)
validation_dataset = RegressionDataset(target=validation_output, spike_in = validation_input)
validation_loader  =  torch.utils.data.DataLoader(dataset=validation_dataset, batch_size = len(validation_input), drop_last=True)
test_dataset = BroadbandDataset(target=test_output, spike_in = test_input)
test_loader =  torch.utils.data.DataLoader(dataset=test_dataset, batch_size = len(test_input), drop_last=False)

for data, target, id in train_loader:
    print("Shape of dati:", data.shape)
    print("Shape ofl target:", target.shape)
    print("Batches in Training :", len(train_loader))
    break  # exit after first batch

#Construct Model


In [13]:
class Net(torch.nn.Module):
    """Simple spiking neural network in snntorch."""

    def __init__(self, timesteps, input = 1, target_variables = 1, testing_broad = False):
        super().__init__()

        l0 = train_setting['hyperparams'][0]
        l1 = train_setting['hyperparams'][1]
        l2 = train_setting['hyperparams'][2]

        self.timesteps = timesteps # number of time steps to simulate the network
        self.testing_broad = testing_broad
        spike_grad = surrogate.fast_sigmoid() # surrogate gradient function

        if train_setting['1thr4layer']:
          thr_in       = train_setting['thr_init']
          thr_hidden   = train_setting['thr_init']
          thr_hidden2  = train_setting['thr_init']
        else:
          thr_in      = torch.ones(l0)*train_setting['thr_init']
          thr_hidden  = torch.ones(l1)*train_setting['thr_init']
          thr_hidden2 = torch.ones(l2)*train_setting['thr_init']

        if train_setting['1beta4layer']:
          beta_in      = train_setting['bet_init']
          beta_hidden  = train_setting['bet_init']
          beta_hidden2 = train_setting['bet_init']
          beta_out     = train_setting['bet_init']
        else:
          beta_in      = torch.ones(l0)*train_setting['bet_init']
          beta_hidden  = torch.ones(l1)*train_setting['bet_init']
          beta_hidden2 = torch.ones(l2)*train_setting['bet_init']
          beta_out     = torch.ones(target_variables)*train_setting['bet_init']
          if train_setting['beta_out_single']:
            beta_out   = train_setting['bet_init']


        # layer 1
        self.fc_in = torch.nn.Linear(in_features=input, out_features=l0, bias = train_setting['bias'])
        self.lif_in = snn.Leaky(beta=beta_in, threshold=thr_in, learn_beta=train_setting['decay_train'], spike_grad=spike_grad, learn_threshold=train_setting['thr_train'], reset_mechanism=train_setting['reset'])

        # layer 2
        self.fc_hidden = torch.nn.Linear(in_features=l0, out_features=l1, bias = train_setting['bias'])
        self.lif_hidden = snn.Leaky(beta=beta_hidden, threshold=thr_hidden, learn_beta=train_setting['decay_train'], spike_grad=spike_grad, learn_threshold=train_setting['thr_train'], reset_mechanism=train_setting['reset'])

        # layer 3
        self.fc_hidden2 = torch.nn.Linear(in_features=l1, out_features=l2, bias = train_setting['bias'])
        self.lif_hidden2 = snn.Leaky(beta=beta_hidden2, threshold=thr_hidden2, learn_beta=train_setting['decay_train'], spike_grad=spike_grad, learn_threshold=train_setting['thr_train'], reset_mechanism=train_setting['reset'])

        # layer 4: leaky integrator neuron. Note the reset mechanism is disabled and we will disregard output spikes.
        self.fc_out = torch.nn.Linear(in_features=l2, out_features = target_variables, bias = train_setting['bias'])
        self.li_out = snn.Leaky(beta=beta_out, threshold=1.0, learn_beta=train_setting['decay_train'], spike_grad=spike_grad, reset_mechanism="none")


    def forward(self, x):
        """Forward pass for several time steps."""

        # Initalize membrane potential
        mem_1 = self.lif_in.init_leaky()
        mem_2 = self.lif_hidden.init_leaky()
        mem_22 = self.lif_hidden2.init_leaky()
        mem_3 = self.li_out.init_leaky()

        # Empty lists to record outputs
        mem_3_rec = []
        # Loop over
        for step in range(self.timesteps):
            x_timestep = x[step, :, :]

            cur_in = self.fc_in(x_timestep)
            spk_in, mem_1 = self.lif_in(cur_in, mem_1)

            cur_hidden = self.fc_hidden(spk_in)
            spk_hidden, mem_2 = self.lif_hidden(cur_hidden, mem_2)

            cur_hidden2 = self.fc_hidden2(spk_hidden)
            spk_hidden2, mem_22 = self.lif_hidden2(cur_hidden2, mem_22)

            cur_out = self.fc_out(spk_hidden2)
            _, mem_3 = self.li_out(cur_out, mem_3)

            mem_3_rec.append(mem_3)

        if self.testing_broad:
          out = torch.stack(mem_3_rec).squeeze()
        else:
          out = torch.stack(mem_3_rec)

        return out


In [ ]:
n_last = 2

if torch.cuda.is_available():
  device = torch.device('cuda')
  print(torch.cuda.get_device_name(0))
else:
  device = torch.device('cpu')
  print('cpu')

steps = training_input.shape[2]

input_neurons = training_input.shape[1]
model = Net(timesteps=steps, input = input_neurons, target_variables = n_last).to(device)

#Construct Training Loop

In [ ]:
num_iter = train_setting['epochs']
optimizer = torch.optim.Adam(params=model.parameters(), lr=train_setting['lr'])
loss_function = torch.nn.MSELoss()

loss_hist   = []
v_loss_hist = []
v_loss_best = 1e10
epoca = 0


# training loop
with tqdm.trange(num_iter) as pbar:
    for _ in pbar:
        train_batch = iter(train_loader)
        minibatch_counter = 0
        loss_epoch = []
        loss_simple_epoch = []
        loss_angular_epoch = []
        loss_pos_epoch = []
        loss_obj_epoch = []
        model.train()
        for feature, label, obj in train_batch:

            feature = feature.to(device)
            label   = label.to(device)

            feature = torch.swapaxes(input=feature, axis0=0, axis1=1)
            label   = torch.swapaxes(input=label, axis0=0, axis1=1)

            mem  = model(feature)
            loss_val = loss_function(mem, label) # calculate loss

            optimizer.zero_grad() # zero out gradients
            loss_val.backward()   # calculate gradients
            optimizer.step()      # update weights

            # store loss
            loss_epoch.append(loss_val.item())
            minibatch_counter += 1
            avg_batch_loss = sum(loss_epoch) / minibatch_counter # calculate average loss p/epoch
            pbar.set_postfix(loss="%.3e" % avg_batch_loss)       # print loss p/batch

        loss_hist.append(avg_batch_loss)
        epoca = epoca + 1
        torch.save(model.state_dict(), trained_folder +  '/network')
        model.eval()
        vloss_sum = 0
        with torch.no_grad():
          for v_feature, v_label, obj in validation_loader:
            feature = feature.to(device)
            v_label   = v_label.to(device)
            v_feature = torch.swapaxes(input=v_feature, axis0=0, axis1=1).to(device)
            v_label   = torch.swapaxes(input=v_label, axis0=0, axis1=1).to(device)
            voutputs = model(v_feature)

            vloss_sum += loss_function(voutputs, v_label)

          vloss = vloss_sum.mean()
          v_loss_hist.append(vloss.cpu())
          if vloss < v_loss_best:
            best_epoch = epoca
            v_loss_best = vloss
            torch.save(model.state_dict(), trained_folder + '/network_best')

        clear_output(wait=True)
        plt.figure(figsize=(10, 6))
        plt.semilogy(loss_hist, label='Training Loss')
        plt.semilogy(v_loss_hist, label = 'Validation Loss')
        plt.title('best epoch {}'.format(best_epoch))
        plt.legend()
        plt.show()

        if (epoca-best_epoch)>patience:
          print("Early exit")
          break

## save final plot in outputpath
plt.figure(figsize=(10, 6))
plt.semilogy(loss_hist, label='Training Loss')
plt.semilogy(v_loss_hist, label = 'Validation Loss')
plt.title('best epoch {}'.format(best_epoch))
plt.legend()
plt.savefig(trained_folder + '/loss.png')

#Evaluation

In [ ]:
input_neurons = test_input.shape[1]
model_test = Net(timesteps=len(test_input), input = input_neurons, target_variables = n_last, testing_broad=True).to(device)
print('Number of input channels',input_neurons)
print(trained_folder)
try:
  model_test.load_state_dict(torch.load(trained_folder + '/network_best'))
except:
  print("An exception occurred")

In [16]:
inference = []

 # pause gradient calculation during evaluation
with torch.no_grad():

    # loop over data samples
    for feature, label in test_loader:
        feature = feature.to(device)
        label   = label.to(device)

        # forward-pass
        mem = model_test(feature)
        inference.append(mem.detach().squeeze().cpu().numpy())

Metrics evaluation on velocity regression

In [ ]:
trained_output = np.array(inference).squeeze()
scale =   400 # use scale 1017.3 for 20170124, 20170127, 20170131
rescale = 1017.3/scale

R = []
Cc = []

for i in range(n_last):
  R.append(Rsquare(test_output[:,i]*rescale, trained_output[:,i]*rescale))
  Cc.append(CC(test_output[:,i]*rescale, trained_output[:,i]*rescale))

print("R^2 [x,y]:", R, "mean:",np.array(R).mean())
print("CC [x,y]:", Cc, "mean:",np.array(Cc).mean())

from sklearn.metrics import mean_absolute_error

print("MAE:",mean_absolute_error(test_output.transpose(1,0).flatten()*rescale,trained_output.transpose(1,0).flatten()*rescale))
print("MAE x:",mean_absolute_error(test_output.transpose(1,0)[0]*rescale,trained_output.transpose(1,0)[0]*rescale))
print("MAE y:",mean_absolute_error(test_output.transpose(1,0)[1]*rescale,trained_output.transpose(1,0)[1]*rescale))


In [ ]:
plt.figure(figsize=(20, 4))
a = 0
b = trained_output.shape[0]
for var_out in range(n_last):

  axs = plt.subplot(1, n_last, var_out + 1)
  axs.plot(trained_output[a:b, var_out]*rescale, label="Inference")
  axs.plot(test_output[a:b, var_out]*rescale, label='GT')
axs.legend()
#plt.savefig(trained_folder + '/velocity_output.png')

Run with NeuroBench code on same testset

In [ ]:
!pip install neurobench

In [ ]:
from neurobench.benchmarks import Benchmark
from neurobench.models.torch_model import TorchModel

workload_metrics = ["r2"]
static_metrics = []
model_bench = TorchModel(model_test)
benchmark = Benchmark(model_bench, test_loader, [], [], [static_metrics, workload_metrics])
results = benchmark.run()
print('Evaluate test set with NeuroBench code')
print(results)

Evaluate Average End Point Error on position estimation at the end of each reach

In [ ]:
refined_target = mat_contents['broadband_dataset'][0][0]['index_var_targets']
my_test_targets = refined_target[training_reaches + validation_reaches:]
my_test_targets = np.array(my_test_targets) - my_test_targets[0]
my_test_targets = my_test_targets.flatten()
print(my_test_targets) ## index of target changes in the test set

In [ ]:
# adjust based on selected record
!wget -O indy_20170123_02.mat "https://zenodo.org/records/3854034/files/indy_20170123_02.mat?download=1"

In [ ]:
import h5py
path = '/content/indy_20170123_02.mat' # adjust based on selected record
with h5py.File(path,'r') as f:
    print(f.keys())
    pos = f['cursor_pos']
    start_pos = np.array(pos).transpose(1,0)[0]
    pos_x = pos[0]
    pos_y = pos[1]
    plt.plot(pos_x,label="cursor x mm")
    plt.show()

print(start_pos)

print(mat_contents['broadband_dataset'][0][0].dtype.names)
vel_ref = mat_contents['broadband_dataset'][0][0]['vel']
input_ref = mat_contents['broadband_dataset'][0][0]['neural_chunks']

initial_position = start_pos*scale
rec_position = np.insert(vel_ref, 0, initial_position, axis=0)
rec_position  = np.cumsum(rec_position,axis=0)*1/scale
rec_position= rec_position.transpose(1,0)

plt.plot(rec_position[0])
plt.show()

rec_position= rec_position.transpose(1,0)


In [ ]:
## test on windowed testset - evaluate velocity error propagation within a single reach

test_input_w = []
test_output_w = []
test_pos_w = []
print(rec_position.shape)
GT_test_pos = rec_position[validation_entries:]
print(GT_test_pos.shape)
initial_pos_test = GT_test_pos[0]*scale
ref_position = np.insert(test_output, 0, initial_pos_test, axis=0)
ref_position  = np.cumsum(ref_position,axis=0)*1/scale
ref_position = ref_position[:-1]
print(ref_position.shape)
print(test_output.shape)


for i in range(len(my_test_targets)-1):
  test_pos_w.append(ref_position[my_test_targets[i]:my_test_targets[i+1]])
  test_input_w.append(test_input[my_test_targets[i]:my_test_targets[i+1]])
  test_output_w.append(test_output[my_test_targets[i]:my_test_targets[i+1]])
test_pos_w.append(ref_position[my_test_targets[-1]:])
test_input_w.append(test_input[my_test_targets[-1]:])
test_output_w.append(test_output[my_test_targets[-1]:])

inference_vel_w = []

for i in range(len(test_input_w)):
  current_input = torch.tensor(test_input_w[i], dtype=torch.float32)
  model_test = Net(timesteps=current_input.shape[0], input = current_input.shape[-1], target_variables = n_last, testing_broad=True).to(device)
  try:
    model_test.load_state_dict(torch.load(trained_folder + '/network_best'))
  except:
    print("An exception occurred")
  model_test.eval()
  current_input = current_input.unsqueeze(1)
  mem = model_test(current_input.to(device))
  inference_vel_w.append(mem.detach().squeeze().cpu().numpy())

inference_pos_w = []
for i in range(len(inference_vel_w)):
  initial_position = test_pos_w[i][0]*scale
  trained_position_c = np.insert(inference_vel_w[i], 0, initial_position, axis=0)[:-1]
  trained_position_c = trained_position_c.transpose()
  inference_pos = np.cumsum(trained_position_c,axis=1).transpose()*1/scale
  inference_pos_w.append(inference_pos)

for i in  range(len(inference_pos_w)):
  if (len(inference_pos_w[i])!=len(test_pos_w[i])):
    print('Len difference is at index',i,'number of items is',len(inference_pos_w))
  if i == 0:
     cont_inference_pos_w = inference_pos_w[i]
     cont_test_pos_w = test_pos_w[i]
  else:
    cont_inference_pos_w = np.concatenate((cont_inference_pos_w, inference_pos_w[i]), axis=0)
    cont_test_pos_w = np.concatenate((cont_test_pos_w, test_pos_w[i]), axis=0)


plt.figure(figsize=(20, 4))
a = 0
b = cont_inference_pos_w.shape[0]
for var_out in range(n_last):
  curr=0
  axs = plt.subplot(n_last, 1, var_out + 1)
  axs.plot(cont_inference_pos_w[a:b, var_out], label="Inference")
  axs.plot(cont_test_pos_w[a:b, var_out], label='GT reset start')
  for i in range(len(inference_pos_w)):
    curr = curr + len(inference_pos_w[i])
    axs.plot([curr,curr],[-50,50],color='red',linewidth=0.2)

axs.legend()
#plt.savefig(trained_folder + '/position_w_output_singlereach_target_final.png')

R_position = []
Cc_position = []

for i in range(2):
  R_position.append(Rsquare(cont_test_pos_w[:,i], cont_inference_pos_w[:,i]))
  Cc_position.append(CC(cont_test_pos_w[:,i], cont_inference_pos_w[:,i]))

print("R^2 [x,y]:", R_position, "mean:",np.array(R_position).mean())
print("CC [x,y]:", Cc_position, "mean:",np.array(Cc_position).mean())

print("R^2 flattened:",Rsquare(cont_test_pos_w.transpose(1,0).flatten(),cont_inference_pos_w.transpose(1,0).flatten()))
print("CC flattened:",CC(cont_test_pos_w.transpose(1,0).flatten(),cont_inference_pos_w.transpose(1,0).flatten()))

print("MAE:",mean_absolute_error(cont_test_pos_w.transpose(1,0).flatten(),cont_inference_pos_w.transpose(1,0).flatten()))
print("MAE x:",mean_absolute_error(cont_test_pos_w.transpose(1,0)[0],cont_inference_pos_w.transpose(1,0)[0]))
print("MAE y:",mean_absolute_error(cont_test_pos_w.transpose(1,0)[1],cont_inference_pos_w.transpose(1,0)[1]))

last_window_errx = []
last_window_erry = []

for i in range(len(inference_pos_w)):
  last_window_errx.append(abs(test_pos_w[i][-1][0]-inference_pos_w[i][-1][0]))
  last_window_erry.append(abs(test_pos_w[i][-1][1]-inference_pos_w[i][-1][1]))


print("check:",len(last_window_errx))

last_window_errx = np.array(last_window_errx)
last_window_erry = np.array(last_window_erry)
epe_error = np.sqrt(last_window_errx**2+last_window_erry**2)

for i in range(len(inference_pos_w)):
  print(f"monkey: ({test_pos_w[i][-1][0]},{test_pos_w[i][-1][1]}) inference: ({inference_pos_w[i][-1][0]},{inference_pos_w[i][-1][1]}) deltax: {last_window_errx[i]} deltay: {last_window_erry[i]} epe: {epe_error[i]}")

print("Avg EPE:",epe_error.mean())
print("Avg EPE x:",last_window_errx.mean())
print("Avg EPE y:",last_window_erry.mean())
